In [11]:
import scanpy as sc
import pandas as pd
import h5py
import numpy as np

# ========================================
# Step 1: Diagnose the H5AD file
# ========================================

h5ad_file = "./data/Nonneuron.h5ad"

print("🔍 Diagnosing H5AD file...")

# Check if file exists and basic info
import os
if os.path.exists(h5ad_file):
    file_size = os.path.getsize(h5ad_file) / (1024**2)  # MB
    print(f"✅ File exists: {file_size:.1f} MB")
else:
    print("❌ File not found!")
    exit()

# Try to inspect with h5py first
try:
    with h5py.File(h5ad_file, 'r') as f:
        print(f"✅ H5 file structure:")
        def print_h5_structure(name, obj):
            if len(name.split('/')) <= 3:  # Limit depth
                print(f"   {name}: {type(obj)}")
        f.visititems(print_h5_structure)
except Exception as e:
    print(f"❌ H5 inspection failed: {e}")

# ========================================
# Step 2: Try different loading methods
# ========================================

print(f"\n🔧 Trying different loading methods...")

# Method 1: Standard scanpy
try:
    adata = sc.read_h5ad(h5ad_file)
    print(f"✅ Method 1 (scanpy) worked!")
    print(f"   Shape: {adata.shape}")
except Exception as e:
    print(f"❌ Method 1 failed: {e}")
    adata = None

# Method 2: Try with backed mode
if adata is None:
    try:
        adata = sc.read_h5ad(h5ad_file, backed='r')
        print(f"✅ Method 2 (backed) worked!")
        print(f"   Shape: {adata.shape}")
    except Exception as e:
        print(f"❌ Method 2 failed: {e}")

# Method 3: Try reading as HDF5 directly
if adata is None:
    try:
        import anndata
        adata = anndata.read_h5ad(h5ad_file)
        print(f"✅ Method 3 (anndata direct) worked!")
        print(f"   Shape: {adata.shape}")
    except Exception as e:
        print(f"❌ Method 3 failed: {e}")

# Method 4: Try older scanpy version compatibility
if adata is None:
    try:
        # Sometimes works with specific chunk settings
        adata = sc.read_h5ad(h5ad_file, chunk_size=6000)
        print(f"✅ Method 4 (chunked) worked!")
        print(f"   Shape: {adata.shape}")
    except Exception as e:
        print(f"❌ Method 4 failed: {e}")

# ========================================
# Step 3: Alternative - Check if it's a different format
# ========================================

if adata is None:
    print(f"\n🔄 Checking for alternative formats...")
    
    # Check if it's actually an h5 file (not h5ad)
    try:
        import scanpy as sc
        adata = sc.read_10x_h5(h5ad_file)
        print(f"✅ File is 10X H5 format!")
        print(f"   Shape: {adata.shape}")
        # Make variable names unique (genes)
        adata.var_names_unique()
    except Exception as e:
        print(f"❌ Not 10X H5 format: {e}")

🔍 Diagnosing H5AD file...
✅ File exists: 15281.3 MB
✅ H5 file structure:
   X: <class 'h5py._hl.group.Group'>
   X/data: <class 'h5py._hl.dataset.Dataset'>
   X/indices: <class 'h5py._hl.dataset.Dataset'>
   X/indptr: <class 'h5py._hl.dataset.Dataset'>
   layers: <class 'h5py._hl.group.Group'>
   obs: <class 'h5py._hl.group.Group'>
   obs/_index: <class 'h5py._hl.dataset.Dataset'>
   obs/abc_sample_id: <class 'h5py._hl.dataset.Dataset'>
   obs/anatomical_division_label_x: <class 'h5py._hl.group.Group'>
   obs/anatomical_division_label_x/categories: <class 'h5py._hl.dataset.Dataset'>
   obs/anatomical_division_label_x/codes: <class 'h5py._hl.dataset.Dataset'>
   obs/anatomical_division_label_y: <class 'h5py._hl.group.Group'>
   obs/anatomical_division_label_y/categories: <class 'h5py._hl.dataset.Dataset'>
   obs/anatomical_division_label_y/codes: <class 'h5py._hl.dataset.Dataset'>
   obs/barcoded_cell_sample_label: <class 'h5py._hl.group.Group'>
   obs/barcoded_cell_sample_label/categor

/home/ubuntu/miniforge/envs/scgpt/lib/python3.10/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


✅ Method 1 (scanpy) worked!
   Shape: (888263, 59357)


/home/ubuntu/miniforge/envs/scgpt/lib/python3.10/site-packages/anndata/_core/anndata.py:1830: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


In [12]:
import scanpy as sc
import pandas as pd
import numpy as np

# Load your h5ad file
#h5ad_file = "./data/BestOneYet.h5ad"
adata = sc.read_h5ad(h5ad_file)  # Replace with your actual filename

print(f"Loaded h5ad file:")
print(f"  Shape: {adata.shape} (cells x genes)")
print(f"  Available obs columns: {list(adata.obs.columns)}")
print(f"  Available var columns: {list(adata.var.columns)}")



/home/ubuntu/miniforge/envs/scgpt/lib/python3.10/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


Loaded h5ad file:
  Shape: (888263, 59357) (cells x genes)
  Available obs columns: ['cell_barcode_x', 'library_label_x', 'anatomical_division_label_x', 'cell_barcode_y', 'barcoded_cell_sample_label', 'library_label_y', 'feature_matrix_label', 'entity', 'brain_section_label', 'library_method', 'donor_label', 'donor_sex', 'dataset_label', 'x', 'y', 'cluster_alias', 'region_of_interest_label', 'anatomical_division_label_y', 'abc_sample_id', 'number_of_cells_x', 'label', 'name', 'cluster_annotation_term_set_label', 'parent_term_label', 'parent_term_set_label', 'term_set_order', 'term_order', 'cluster_annotation_term_set_name', 'color_hex_triplet', 'number_of_cells_y', 'description']
  Available var columns: ['gene_symbol']


/home/ubuntu/miniforge/envs/scgpt/lib/python3.10/site-packages/anndata/_core/anndata.py:1830: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


In [13]:
# ========================================
# Extract Gene Names 
# ========================================

# Convert to list first to avoid any DataFrame column naming
gene_names_list = adata.var_names.tolist()
gene_names_df = pd.DataFrame(gene_names_list)

# Save gene names (without index, header=False to match your format)
gene_names_df.to_csv("gene_names.csv", index=False, header=False)
print(f"\n✅ Saved gene_names.csv: {len(gene_names_df)} genes")


✅ Saved gene_names.csv: 59357 genes


In [14]:
# ========================================
# UNIVERSAL METADATA EXTRACTION
# Works for Uniklinik, ABC Atlas, or any dataset
# ========================================

def extract_and_save_metadata(adata, dataset_type='auto', output_path='metadata.csv'):
    """
    Extract metadata from AnnData and save to CSV
    Automatically adapts to different dataset formats
    
    Args:
        adata: AnnData object
        dataset_type: 'uniklinik', 'abc_atlas', or 'auto'
        output_path: Where to save CSV
    """
    
    import pandas as pd
    import numpy as np
    
    print("="*80)
    print("EXTRACTING METADATA")
    print("="*80)
    
    # Start with observation metadata
    metadata_df = adata.obs.copy()
    metadata_df['barcode'] = metadata_df.index
    
    # ========================================
    # ADD UMAP COORDINATES (UNIVERSAL)
    # ========================================
    
    print("\n📊 Looking for UMAP coordinates...")
    umap_found = False
    
    # Check adata.obsm
    if 'X_umap' in adata.obsm:
        print("  ✅ Found in adata.obsm['X_umap']")
        umap_coords = adata.obsm['X_umap']
        metadata_df['UMAP_1'] = umap_coords[:, 0]
        metadata_df['UMAP_2'] = umap_coords[:, 1]
        umap_found = True
    
    elif 'umap' in adata.obsm:
        print("  ✅ Found in adata.obsm['umap']")
        umap_coords = adata.obsm['umap']
        metadata_df['UMAP_1'] = umap_coords[:, 0]
        metadata_df['UMAP_2'] = umap_coords[:, 1]
        umap_found = True
    
    # Check adata.obs (ABC Atlas style)
    elif 'x' in metadata_df.columns and 'y' in metadata_df.columns:
        print("  ✅ Found in adata.obs['x'], ['y'] (ABC Atlas format)")
        metadata_df['UMAP_1'] = metadata_df['x']
        metadata_df['UMAP_2'] = metadata_df['y']
        umap_found = True
    
    else:
        print("  ⚠️  No UMAP found")
    
    # ========================================
    # DETECT DATASET TYPE
    # ========================================
    
    if dataset_type == 'auto':
        print("\n🔍 Auto-detecting dataset type...")
        
        if 'CellTypes' in metadata_df.columns and 'seurat_clusters' in metadata_df.columns:
            dataset_type = 'uniklinik'
            print("  ✅ Detected: UNIKLINIK")
        
        elif 'name' in metadata_df.columns and 'cluster_alias' in metadata_df.columns:
            dataset_type = 'abc_atlas'
            print("  ✅ Detected: ABC ATLAS")
        
        else:
            dataset_type = 'generic'
            print("  ⚠️  Unknown format, using generic column order")
    
    # ========================================
    # DEFINE PREFERRED COLUMNS
    # ========================================
    
    if dataset_type == 'uniklinik':
        preferred_columns = [
            'barcode', 'CellTypes', 'orig.ident', 
            'nCount_RNA', 'nFeature_RNA', 'seurat_clusters',
            'UMAP_1', 'UMAP_2'
        ]
    
    elif dataset_type == 'abc_atlas':
        preferred_columns = [
            'barcode', 'celltype', 'celltype_id',
            'name', 'subtype_id', 'description', 'cluster_alias',
            'UMAP_1', 'UMAP_2',
            'donor_label', 'donor_sex', 'anatomical_division_label_y',
            'brain_section_label', 'parent_term_label'
        ]
    
    else:  # generic
        preferred_columns = [
            'barcode', 'UMAP_1', 'UMAP_2'
        ]
    
    # ========================================
    # REORDER COLUMNS
    # ========================================
    
    # Keep only existing preferred columns
    existing_preferred = [col for col in preferred_columns if col in metadata_df.columns]
    
    # Get other columns
    other_columns = [col for col in metadata_df.columns if col not in existing_preferred]
    
    # Final order
    final_columns = existing_preferred + other_columns
    metadata_df = metadata_df[final_columns]
    
    # ========================================
    # SAVE
    # ========================================
    
    metadata_df.to_csv(output_path, index=False)
    
    print(f"\n✅ SAVED: {output_path}")
    print(f"   Dataset type: {dataset_type}")
    print(f"   Cells: {len(metadata_df):,}")
    print(f"   Total columns: {len(metadata_df.columns)}")
    print(f"   Priority columns: {len(existing_preferred)}")
    
    # Show first columns
    print(f"\n📋 First 8 columns:")
    for i, col in enumerate(metadata_df.columns[:8], 1):
        print(f"   {i}. {col}")
    
    # Show sample
    print(f"\n📊 Sample data:")
    display_cols = metadata_df.columns[:5].tolist()
    print(metadata_df[display_cols].head(3))
    
    print("="*80)
    
    return metadata_df

# ========================================
# USAGE
# ========================================

# Auto-detect and save
metadata = extract_and_save_metadata(adata)

# OR specify dataset type
metadata = extract_and_save_metadata(adata, dataset_type='abc_atlas')

# OR specify custom output path
metadata = extract_and_save_metadata(
    adata, 
    dataset_type='uniklinik',
    output_path='./uniklinik_metadata.csv'
)

  ⚠️  No UMAP coordinates found
✅ Saved metadata.csv: 888263 cells, 32 columns


In [15]:
import pandas as pd

# Load the metadata
metadata_df = pd.read_csv('metadata.csv')

# Rename the column
#metadata_df.rename(columns={'celltype_final': 'CellTypes'}, inplace=True)

# Save the updated metadata
metadata_df.to_csv('metadata.csv', index=False)
print(f"✅ Saved metadata.csv: {len(metadata_df)} cells, {len(metadata_df.columns)} columns")

print("Column renamed successfully!")
print(f"\nNew columns: {metadata_df.columns.tolist()}")
print(f"\nUnique cell types in dataset: {metadata_df['CellTypes'].nunique()}")
print(f"\nCell type distribution:\n{metadata_df['CellTypes'].value_counts()}")

✅ Saved metadata.csv: 888263 cells, 32 columns
Column renamed successfully!

New columns: ['barcode', 'cell_barcode_x', 'library_label_x', 'anatomical_division_label_x', 'cell_barcode_y', 'barcoded_cell_sample_label', 'library_label_y', 'feature_matrix_label', 'entity', 'brain_section_label', 'library_method', 'donor_label', 'donor_sex', 'dataset_label', 'x', 'y', 'cluster_alias', 'region_of_interest_label', 'anatomical_division_label_y', 'abc_sample_id', 'number_of_cells_x', 'label', 'name', 'cluster_annotation_term_set_label', 'parent_term_label', 'parent_term_set_label', 'term_set_order', 'term_order', 'cluster_annotation_term_set_name', 'color_hex_triplet', 'number_of_cells_y', 'description']


KeyError: 'CellTypes'

In [10]:
# ========================================
# Summary and Verification
# ========================================

print(f"\n📊 Summary:")
print(f"  Gene names: {len(gene_names_df)} genes saved")
print(f"  Metadata: {len(metadata_df)} cells with {len(metadata_df.columns)} columns")

print(f"\n📋 Metadata columns:")
for i, col in enumerate(metadata_df.columns, 1):
    print(f"  {i:2d}. {col}")

# Show sample of each file
print(f"\n🔍 Sample gene_names.csv (first 5):")
print(gene_names_df.head())

print(f"\n🔍 Sample metadata.csv (first 3 rows, key columns):")
key_cols = ['barcode', 'CellTypes'] + [col for col in ['UMAP_1', 'UMAP_2', 'seurat_clusters'] if col in metadata_df.columns]
if len(key_cols) > 0:
    print(metadata_df[key_cols].head(3))
else:
    print(metadata_df.head(3))

print(f"\n Files ready for use with scGPT!")


📊 Summary:
  Gene names: 66587 genes saved
  Metadata: 48636 cells with 23 columns

📋 Metadata columns:
   1. barcode
   2. orig.ident
   3. nCount_RNA
   4. nFeature_RNA
   5. seurat_clusters
   6. UMAP_1
   7. UMAP_2
   8. nCount_SCT
   9. nFeature_SCT
  10. patient
  11. Suva_A
  12. cycling
  13. region
  14. leiden1
  15. celltype
  16. percent.mt
  17. SCT_snn_res.0.8
  18. integrated_snn_res.0.5
  19. proportion_dupli_chr7
  20. proportion_loss_chr10
  21. region_final
  22. malignancy
  23. CellTypes

🔍 Sample gene_names.csv (first 5):
             0
0  MIR1302-2HG
1   AL627309.1
2   AL627309.5
3    LINC01409
4    LINC01128

🔍 Sample metadata.csv (first 3 rows, key columns):
                          barcode CellTypes    UMAP_1    UMAP_2  \
0  PT90_Pink-AAACGCTAGGAAACGA-1_1    Neuron  7.230384  5.666156   
1  PT90_Pink-AAATGGAGTTGCTCGG-1_1       GBM  4.888780 -7.170683   
2  PT90_Pink-AACCCAATCCCAAGTA-1_1       GBM  6.205311 -8.147423   

   seurat_clusters  
0                